# "But wait... there's more"

## A More Visible Agent Loop

The Digital Twin contained an Agent Loop. But it was behind-the-scenes, running every time the user asked a message. Using its tools and then replying. It didn't feel very... loopy.

### Adding 2 more ingredients to make it more real

Let's make an Agent Loop with some familiar features borrowed from Claude Code:

1. A Terminal UI (TUI)
2. A Checklist tool to cause and track multiple tool calls


In [1]:
# Start with some imports - rich is a library for making formatted text output in the terminal

from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
import json
import os
import logging
load_dotenv(override=True)

True

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("twin.loop")

def _pretty(obj):
    """Biztonságos, olvasható JSON-formázás a logokhoz."""
    try:
        return json.dumps(obj, ensure_ascii=False, indent=2, default=str)
    except Exception:
        return repr(obj)

In [19]:
def _dump_messages(messages, cim="messages"):
    """A teljes messages lista olvasható kiírása, objektum és dict elemekkel is."""
    log.info("%s tartalma (%d elem):", cim, len(messages))
    for i, m in enumerate(messages):
        # OpenAI message-objektumot dict-té alakítunk, a dict marad dict
        if hasattr(m, "model_dump"):
            d = m.model_dump()
        elif isinstance(m, dict):
            d = m
        else:
            d = {"repr": repr(m)}
        role = d.get("role", "?")
        log.info("  [%d] role=%s", i, role)
        log.debug("      %s", _pretty(d))

In [3]:
def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)

In [4]:
"""openai = OpenAI(
    api_key=os.getenv("API_KEY"),      # "ollama" – dummy, de nem lehet üres
    base_url=os.getenv("BASE_URL"),    # http://localhost:11434/v1  ← ez irányít az Ollamára
)"""
model = os.getenv("MODEL")
openai = OpenAI()

In [5]:
# Some lists!

checklist = []
completed = []

In [6]:
def get_checklist_report() -> str:
    result = ""
    for index, item in enumerate(checklist):
        if completed[index]:
            result += f"Checklist #{index + 1}: [green][strike]{item}[/strike][/green]\n"
        else:
            result += f"Checklist #{index + 1}: {item}\n"
    show(result)
    return result

In [7]:
get_checklist_report()

''

In [8]:
def create_checklist(descriptions: list[str]) -> str:
    checklist.extend(descriptions)
    completed.extend([False] * len(descriptions))
    return get_checklist_report()

In [9]:
def mark_complete(index: int, completion_notes: str) -> str:
    if 1 <= index <= len(checklist):
        completed[index - 1] = True
    else:
        return "No checklist at this index."
    Console().print(completion_notes)
    return get_checklist_report()

In [10]:
checklist, completed = [], []

create_checklist(["Buy groceries", "Finish week 1", "Eat banana"])

Checklist #1: Buy groceries
Checklist #2: Finish week 1
Checklist #3: Eat banana

'Checklist #1: Buy groceries\nChecklist #2: Finish week 1\nChecklist #3: Eat banana\n'

In [11]:
mark_complete(1, "bought")

bought

Checklist #1: Buy groceries
Checklist #2: Finish week 1
Checklist #3: Eat banana

'Checklist #1: [green][strike]Buy groceries[/strike][/green]\nChecklist #2: Finish week 1\nChecklist #3: Eat banana\n'

In [12]:
create_checklist_json = {
    "name": "create_checklist",
    "description": "Add new checklist from a list of descriptions and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "descriptions": {
                'type': 'array',
                'items': {'type': 'string'},
                'title': 'Descriptions of checklist items'
                }
            },
        "required": ["descriptions"],
        "additionalProperties": False
    }
}

In [13]:
mark_complete_json = {
    "name": "mark_complete",
    "description": "Mark complete the checklist item at the given position (starting from 1) and return the full list",
    "parameters": {
        'properties': {
            'index': {
                'description': 'The 1-based index of the checklist item to mark as complete',
                'title': 'Index',
                'type': 'integer'
                },
            'completion_notes': {
                'description': 'Notes about how you completed the checklist item in rich console markup',
                'title': 'Completion Notes',
                'type': 'string'
                }
            },
        'required': ['index', 'completion_notes'],
        'type': 'object',
        'additionalProperties': False
    }
}

In [14]:
tools = [{"type": "function", "function": create_checklist_json},
        {"type": "function", "function": mark_complete_json}]

In [15]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [ ]:
"""def loop(messages):
    response = openai.chat.completions.create(model=model, messages=messages, tools=tools)
    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        tool_calls = message.tool_calls
        results = handle_tool_calls(tool_calls)
        messages.append(message)
        messages.extend(results)
        response = openai.chat.completions.create(model=model, messages=messages, tools=tools)
    show(response.choices[0].message.content)"""

In [20]:
def loop(messages):
    log.info("=" * 60)
    log.info("loop() START – bejövő messages db: %d", len(messages))
    for i, m in enumerate(messages):
        role = m.get("role") if isinstance(m, dict) else getattr(m, "role", "?")
        log.debug("  bejövő[%d] role=%s tartalom=%s", i, role, _pretty(m))

    log.info("OpenAI hívás #0 (első kérés) – model=%s, tools db=%d", model, len(tools))
    response = openai.chat.completions.create(model=model, messages=messages, tools=tools)
    log.debug("Nyers válasz #0: %s", _pretty(response.model_dump()))

    iteration = 0
    while response.choices[0].finish_reason == "tool_calls":
        iteration += 1
        log.info("-" * 60)
        log.info("ITERÁCIÓ #%d – finish_reason=tool_calls", iteration)

        message = response.choices[0].message
        tool_calls = message.tool_calls

        log.info("Asszisztens üzenet content=%r, kért tool-ok db=%d",
                 message.content, len(tool_calls or []))

        # Minden kért tool-hívás kiírása (név + argumentumok)
        for j, tc in enumerate(tool_calls or []):
            log.info("  tool_call[%d]: id=%s függvény=%s", j, tc.id, tc.function.name)
            log.debug("    argumentumok (nyers): %s", tc.function.arguments)
            try:
                parsed = json.loads(tc.function.arguments or "{}")
                log.debug("    argumentumok (parsed): %s", _pretty(parsed))
            except json.JSONDecodeError as e:
                log.warning("    argumentum JSON-hiba: %s", e)

        # Tool-ok lefuttatása
        log.info("handle_tool_calls() hívása %d tool-ra...", len(tool_calls or []))
        results = handle_tool_calls(tool_calls)
        log.info("handle_tool_calls() visszatért %d eredménnyel", len(results))
        for k, r in enumerate(results):
            log.debug("  tool eredmény[%d]: %s", k, _pretty(r))

        # Előzmény bővítése
        messages.append(message)
        messages.extend(results)
        log.info("messages mérete az iteráció után: %d", len(messages))
        _dump_messages(messages, cim=f"messages az iteráció #{iteration} után")

        # Következő kör
        log.info("OpenAI hívás #%d (tool eredményekkel)...", iteration)
        response = openai.chat.completions.create(model=model, messages=messages, tools=tools)
        log.debug("Nyers válasz #%d: %s", iteration, _pretty(response.model_dump()))

    log.info("-" * 60)
    log.info("CIKLUS VÉGE – összes iteráció: %d, végső finish_reason=%s",
             iteration, response.choices[0].finish_reason)
    final = response.choices[0].message.content
    log.info("VÉGSŐ válasz content: %r", final)
    log.info("loop() END")
    log.info("=" * 60)

    show(final)

In [23]:
system_message = """
You are given a problem to solve, by using your checklist tools to plan a list of steps, then carrying out each step in turn.
Now create a plan, set the checklist, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""
user_message = """"
A train leaves Boston at 2:00 pm traveling 60 mph.
Another train leaves New York at 3:00 pm traveling 80 mph toward Boston.
When do they meet?
"""
messages = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]

In [24]:
checklist, completed = [], []
loop(messages)

09:12:52 | INFO    | ============================================================
09:12:52 | INFO    | loop() START – bejövő messages db: 2
09:12:52 | INFO    | OpenAI hívás #0 (első kérés) – model=gpt-5.5, tools db=2
09:12:54 | INFO    | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
09:12:54 | INFO    | ------------------------------------------------------------
09:12:54 | INFO    | ITERÁCIÓ #1 – finish_reason=tool_calls
09:12:54 | INFO    | Asszisztens üzenet content=None, kért tool-ok db=1
09:12:54 | INFO    |   tool_call[0]: id=call_CbOWIntkiP3hdnKvKRUqZQ8q függvény=create_checklist
09:12:54 | INFO    | handle_tool_calls() hívása 1 tool-ra...


Checklist #1: Identify known quantities and assumptions, including the Boston–New York distance if not provided.
Checklist #2: Compute Boston train’s head start distance by 3:00 pm.
Checklist #3: Compute remaining distance between trains at 3:00 pm.
Checklist #4: Use combined speed to find travel time until meeting.
Checklist #5: Convert elapsed time into clock time and present the answer.

09:12:54 | INFO    | handle_tool_calls() visszatért 1 eredménnyel
09:12:54 | INFO    | messages mérete az iteráció után: 4
09:12:54 | INFO    | messages az iteráció #1 után tartalma (4 elem):
09:12:54 | INFO    |   [0] role=system
09:12:54 | INFO    |   [1] role=user
09:12:54 | INFO    |   [2] role=assistant
09:12:54 | INFO    |   [3] role=tool
09:12:54 | INFO    | OpenAI hívás #1 (tool eredményekkel)...
09:12:57 | INFO    | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
09:12:57 | INFO    | ------------------------------------------------------------
09:12:57 | INFO    | ITERÁCIÓ #2 – finish_reason=tool_calls
09:12:57 | INFO    | Asszisztens üzenet content=None, kért tool-ok db=1
09:12:57 | INFO    |   tool_call[0]: id=call_uawkMw6t5DJ4CYdxGJVjFZOp függvény=mark_complete
09:12:57 | INFO    | handle_tool_calls() hívása 1 tool-ra...


Known: Boston train leaves at 2:00 pm at 60 mph; New York train leaves at 3:00 pm at 80 mph. Distance is not 
provided, so use a reasonable Boston–New York rail/road distance estimate of about 215 miles.

Checklist #1: Identify known quantities and assumptions, including the Boston–New York distance if not provided.
Checklist #2: Compute Boston train’s head start distance by 3:00 pm.
Checklist #3: Compute remaining distance between trains at 3:00 pm.
Checklist #4: Use combined speed to find travel time until meeting.
Checklist #5: Convert elapsed time into clock time and present the answer.

09:12:57 | INFO    | handle_tool_calls() visszatért 1 eredménnyel
09:12:57 | INFO    | messages mérete az iteráció után: 6
09:12:57 | INFO    | messages az iteráció #2 után tartalma (6 elem):
09:12:57 | INFO    |   [0] role=system
09:12:57 | INFO    |   [1] role=user
09:12:57 | INFO    |   [2] role=assistant
09:12:57 | INFO    |   [3] role=tool
09:12:57 | INFO    |   [4] role=assistant
09:12:57 | INFO    |   [5] role=tool
09:12:57 | INFO    | OpenAI hívás #2 (tool eredményekkel)...
09:13:00 | INFO    | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
09:13:00 | INFO    | ------------------------------------------------------------
09:13:00 | INFO    | ITERÁCIÓ #3 – finish_reason=tool_calls
09:13:00 | INFO    | Asszisztens üzenet content=None, kért tool-ok db=1
09:13:00 | INFO    |   tool_call[0]: id=call_26a4HUXqDODQauAhZK3iviGX függvény=mark_complete
09:13:00 | INFO    | handle_tool_calls() hívása 1 tool-ra...


... (9 outputs left out)

Adding about 1 hour 6 minutes to 3:00 pm gives approximately 4:06 pm.

Checklist #1: Identify known quantities and assumptions, including the Boston–New York distance if not provided.
Checklist #2: Compute Boston train’s head start distance by 3:00 pm.
Checklist #3: Compute remaining distance between trains at 3:00 pm.
Checklist #4: Use combined speed to find travel time until meeting.
Checklist #5: Convert elapsed time into clock time and present the answer.

09:13:04 | INFO    | handle_tool_calls() visszatért 1 eredménnyel
09:13:04 | INFO    | messages mérete az iteráció után: 14
09:13:04 | INFO    | messages az iteráció #6 után tartalma (14 elem):
09:13:04 | INFO    |   [0] role=system
09:13:04 | INFO    |   [1] role=user
09:13:04 | INFO    |   [2] role=assistant
09:13:04 | INFO    |   [3] role=tool
09:13:04 | INFO    |   [4] role=assistant
09:13:04 | INFO    |   [5] role=tool
09:13:04 | INFO    |   [6] role=assistant
09:13:04 | INFO    |   [7] role=tool
09:13:04 | INFO    |   [8] role=assistant
09:13:04 | INFO    |   [9] role=tool
09:13:04 | INFO    |   [10] role=assistant
09:13:04 | INFO    |   [11] role=tool
09:13:04 | INFO    |   [12] role=assistant
09:13:04 | INFO    |   [13] role=tool
09:13:04 | INFO    | OpenAI hívás #6 (tool eredményekkel)...
09:13:07 | INFO    | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
09:13:07 | INFO    | ------------------------------------------------------------
09:13:07

Answer: Approximately 4:06 pm.

Since the Boston–New York distance wasn’t provided, I estimated it as about 215 miles.

Calculation:

• Boston train’s head start:  
1 hour × 60 mph = 60 miles

• Remaining distance at 3:00 pm:  
215 − 60 = 155 miles

• Combined speed:  
60 + 80 = 140 mph

• Time after 3:00 pm:  
155 ÷ 140 ≈ 1.107 hours ≈ 1 hour 6 minutes

So they meet at about 4:06 pm.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Now try to build an Agent Loop from scratch yourself!<br/>
            Create a new .ipynb and make one from first principles, referring back to this as needed.<br/>
            It's one of the few times that I recommend typing from scratch - it's a very satisfying result.
            </span>
        </td>
    </tr>
</table>